In [3]:
import sys
from pathlib import Path
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import torch
from torch.utils.data import DataLoader
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn_v2,
    FasterRCNN_ResNet50_FPN_V2_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

In [4]:
from helpers.utils import set_seed
from helpers.datahelperrr import FireDataset, collate_fn

In [ ]:
DATA_DIR = PROJECT_ROOT / "data"
RUNS = PROJECT_ROOT / "models" / "runs"
NUM_CLASSES = 3  # in torchvision background is always 0

In [6]:
torch.cuda.is_available()

True

In [7]:
set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [9]:
train_ds = FireDataset(DATA_DIR / "train", augment=False)
loader = DataLoader(train_ds, batch_size=2, shuffle=True, collate_fn=collate_fn)

In [10]:
images, targets = next(iter(loader))

In [11]:
images[0].shape, targets[0]["boxes"].shape, targets[0]["labels"]

(torch.Size([3, 640, 640]), torch.Size([1, 4]), tensor([2]))

We use Faster R-CNN ResNet-50-FPN v2 with COCO pretrained weights. After loading we replace only the box predictor so the head outputs fire and smoke. The backbone and FPN stay pretrained.

In [ ]:
def build_fasterrcnn(num_classes = NUM_CLASSES, trainable_backbone_layers = 3):
    weights = FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1
    model = fasterrcnn_resnet50_fpn_v2(weights=weights, trainable_backbone_layers=trainable_backbone_layers) # Right now head is same as COCO pretrained model. 91 class. It can conflict so we dont give the num classes parameter 
    in_features = model.roi_heads.box_predictor.cls_score.in_features # Read how many features go into old head so new head must use the same width
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes) # new head
    model.transform.min_size = (640,)
    model.transform.max_size = 640
    # lock the model's own resize to 640. We already letterbox to 640×640 so this stops it from resizing the image again.
    return model

In [14]:
model = build_fasterrcnn()
model.to(device)

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth" to C:\Users\Arda/.cache\torch\hub\checkpoints\fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth


100%|██████████| 167M/167M [00:22<00:00, 7.68MB/s] 


FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(640,), max_size=640, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, t

In [15]:
model.roi_heads.box_predictor.cls_score.out_features

3

In train mode the model returns a loss dict and in eval mode it returns boxes, labels, and scores. If this cell runs the pipeline is ready for a real training loop

In [16]:
images = [img.to(device) for img in images]
targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

model.train()
loss_dict = model(images, targets)

In [18]:
loss_dict

{'loss_classifier': tensor(1.1459, device='cuda:0', grad_fn=<NllLossBackward0>),
 'loss_box_reg': tensor(0.0332, device='cuda:0', grad_fn=<DivBackward0>),
 'loss_objectness': tensor(0.0577, device='cuda:0',
        grad_fn=<BinaryCrossEntropyWithLogitsBackward0>),
 'loss_rpn_box_reg': tensor(0.0618, device='cuda:0', grad_fn=<DivBackward0>)}

In [19]:
loss_dict.values()

dict_values([tensor(1.1459, device='cuda:0', grad_fn=<NllLossBackward0>), tensor(0.0332, device='cuda:0', grad_fn=<DivBackward0>), tensor(0.0577, device='cuda:0',
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>), tensor(0.0618, device='cuda:0', grad_fn=<DivBackward0>)])

In [22]:
model.eval()
with torch.no_grad():
    preds = model(images)
preds[0].keys()

dict_keys(['boxes', 'labels', 'scores'])

In [23]:
len(preds[0]["boxes"])

100

We are ready to train our model. One training epoch. The model returns all four losses we sum them call backward, and step the optimizer. We start with SGD as in the torchvision detection tutorial. Batch size is 2 because this model is very heavy

In [24]:
from helpers.utils import make_run_dir, BestCheckpoint, append_metrics

train_ds = FireDataset(DATA_DIR / "train", augment=True)
val_ds = FireDataset(DATA_DIR / "val", augment=False)
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=2, shuffle=False, collate_fn=collate_fn)